# Collect Cosmos-Policy contrastive input pairs (both drives): distractors-present vs distractors-absent

Builds **paired** policy-model inputs for downstream contrastive-direction
analysis (LQR / SVD). Single prompt, two scene compositions for `libero_10`
task `0`; within each pair, proprio is held fixed so the only thing that
varies is the camera content (distractor objects on the table).

Prompt (single, used everywhere):

  `"put both the milk and the tomato sauce in the basket"`

Two paired-rollout campaigns:

1. **Negative-drives** — `env_neg` (full 8-object scene) steps; at each
   inference call we snapshot `env_neg.get_sim_state()` and inject it into
   `env_pos` (via `POSITIVE_STRESS.set_init_state`, which slices out the 5
   removed-object qpos/qvel slots). Captures deployment-distribution poses.

2. **Positive-drives** — `env_pos` (milk_only scene) steps; at each inference
   call we snapshot `env_pos.get_sim_state()` and **expand** it back to the
   8-object layout: robot + kept-object qpos/qvel come from `env_pos`'s
   current state; the 5 removed-object qpos/qvel slots are pulled from the
   episode's original `init_state` (they stay at their initial poses because
   they aren't in `env_pos` to move). Captures uncluttered-policy poses, so
   the contrastive direction is fit across the pose region the steered model
   will drift into.

Both LIBERO envs route `set_init_state` through `regenerate_obs_from_state`
([env_wrapper.py:139](work/nvme/bhde/jhong7/LIBERO_pkg/libero/libero/envs/env_wrapper.py#L139)),
which runs `set_state_from_flattened` → `sim.forward()` →
`_update_observables(force=True)` → `_get_observations()` — so the returned
obs has freshly-rendered cameras at the injected pose every time. No camera
staleness.

Pairing: each row in `positive.npz` is paired with the same row in
`negative.npz` — identical proprio, only image content differs. A
`drive_source` column (0=neg-drives, 1=pos-drives) marks which campaign
each row came from.

Caveat on the positive-drives expansion: in `env_pos`, removed distractors
never physically interact with the robot. So when we render them back into
`env_neg` for the contrastive image, they sit at their initial poses even
if the robot has visually passed through where they'd be. That's an
acceptable artifact for "distractors present in image" semantics — the
contrastive direction is about visual content, not physical consistency.

Output layout:

```
notebooks/lqr/inputs/policy_inputs/libero_10__task00__milk_pos_neg/
  positive.npz   # distractors-absent renders at every captured pose
  negative.npz   # distractors-present renders at the same poses
  manifest.json  # prompt, both stress configs, per-rollout summaries
  derived__<slug>.bddl   # BDDL emitted by SceneRemoveObjects
```

In [ ]:
# notebooks/_setup.py lives two dirs up (notebooks/lqr/inputs/<this>.ipynb).
import sys; sys.path.insert(0, '../..')
from _setup import setup_env
setup_env()

import os
os.environ.setdefault('MUJOCO_GL', 'egl')
os.environ.setdefault('PYOPENGL_PLATFORM', 'egl')

In [ ]:
import copy
import hashlib
import json
import re
import time
from collections import deque
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional, Tuple

import numpy as np

from libero.libero import benchmark, get_libero_path
from cosmos_policy.experiments.robot.libero.libero_utils import (
    get_libero_env, get_libero_dummy_action,
)
from cosmos_policy.experiments.robot.libero.run_libero_eval import (
    PolicyEvalConfig, prepare_observation, TASK_MAX_STEPS,
)
from cosmos_policy.experiments.robot.cosmos_utils import (
    get_action, get_model, load_dataset_stats, init_t5_text_embeddings_cache,
)

## 1. `SceneRemoveObjects` (copied from `stress_test/02_remove_objects_milk.ipynb`)

Strips a list of object keys from the BDDL, rewrites the `:goal` so success
still scores against the surviving targets, and truncates LIBERO's saved
init_state to match the smaller body list. Used for two purposes here:
- Once at setup, to build `env_pos` from the stripped BDDL (`transform_task`).
- During negative-drives, to slice the full neg-scene sim state down to the
  positive-scene's smaller `(robot, kept_objects)` layout before injecting
  into `env_pos` (`set_init_state`).

A symmetric helper `_expand_pos_state_to_neg` (cell below) handles the
inverse: turn a pos-scene state vector back into a neg-scene state vector
by pulling removed-object slots from the episode's `init_state`.

In [ ]:
# ---------- BDDL editing helpers ----------

def _find_section_bounds(bddl, keyword):
    """Return (start, end) of the `(:<keyword> ...)` block, paren-balanced, or None."""
    needle = f'(:{keyword}'
    start = bddl.find(needle)
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(bddl)):
        c = bddl[i]
        if c == '(':
            depth += 1
        elif c == ')':
            depth -= 1
            if depth == 0:
                return start, i + 1
    return None


def _strip_paren_block(text, opening_token):
    """Find the first '(<opening_token>' and remove from there through the matching ')'
    plus trailing whitespace + newline. Returns text unchanged if not found."""
    idx = text.find(opening_token)
    if idx == -1:
        return text
    depth = 0
    end = None
    for i in range(idx, len(text)):
        c = text[i]
        if c == '(':
            depth += 1
        elif c == ')':
            depth -= 1
            if depth == 0:
                end = i + 1
                break
    if end is None:
        return text
    while end < len(text) and text[end] in ' \t':
        end += 1
    if end < len(text) and text[end] == '\n':
        end += 1
    line_start = text.rfind('\n', 0, idx) + 1
    if text[line_start:idx].strip() == '':
        idx = line_start
    return text[:idx] + text[end:]


def _drop_section_lines(bddl, keyword, predicate):
    """Within (:<keyword> ...) block, drop any line where predicate(stripped_line) is True."""
    bounds = _find_section_bounds(bddl, keyword)
    if bounds is None:
        return bddl
    s, e = bounds
    section = bddl[s:e]
    keep = []
    for line in section.split('\n'):
        if predicate(line.strip()):
            continue
        keep.append(line)
    return bddl[:s] + '\n'.join(keep) + bddl[e:]


def _parse_objects_order(bddl):
    """Return the list of object keys in :objects declaration order."""
    bounds = _find_section_bounds(bddl, 'objects')
    if bounds is None:
        raise ValueError('BDDL missing :objects section')
    s, e = bounds
    section = bddl[s:e]
    keys = []
    for line in section.split('\n'):
        stripped = line.strip()
        if not stripped or stripped.startswith('(:') or stripped == ')':
            continue
        m = re.match(r'(\S+)\s*-\s*\S+', stripped)
        if m:
            keys.append(m.group(1))
    return keys


def _resolve_libero_problem_obj(env):
    """Walk env.env.env... up to depth 8 looking for the LIBERO problem object."""
    cur, seen = env, set()
    for _ in range(8):
        if cur is None or id(cur) in seen:
            break
        seen.add(id(cur))
        if hasattr(cur, 'objects_dict') and hasattr(cur, 'fixtures_dict'):
            return cur
        cur = getattr(cur, 'env', None)
    return None


# ---------- the stress test ----------

class StressTest:
    """No-op baseline. Subclass and override hooks."""
    slug: str = 'stock'
    def transform_task(self, task, output_dir=None): return task
    def set_init_state(self, env, init_state, episode_idx=0): return env.set_init_state(init_state)
    def apply_to_env(self, env, episode_idx=0) -> None: return None
    def transform_task_desc(self, desc, env=None): return desc
    def manifest(self) -> dict: return {'kind': type(self).__name__, 'slug': self.slug}


@dataclass
class SceneRemoveObjects(StressTest):
    """Strip listed object keys from the BDDL, rewrite the :goal, rewrite the prompt.

    See stress_test/02_remove_objects_milk.ipynb for the original docstring with
    the full mechanism description.
    """
    remove: Tuple[str, ...] = ()
    goal_substitutions: Tuple[Tuple[str, str], ...] = ()
    prompt_replace_mapping: Tuple[Tuple[str, str], ...] = ()
    name_hint: str = 'remove'
    _bddl_path: Optional[str] = field(default=None, init=False, repr=False)
    _orig_order: Tuple[str, ...] = field(default=(), init=False, repr=False)
    _kept_order: Tuple[str, ...] = field(default=(), init=False, repr=False)

    @property
    def slug(self) -> str:
        payload = '|'.join(sorted(self.remove))
        payload += '||' + '|'.join(f'{a}=>{b}' for a, b in self.goal_substitutions)
        payload += '||' + '|'.join(f'{a}=>{b}' for a, b in self.prompt_replace_mapping)
        h = hashlib.md5(payload.encode()).hexdigest()[:6]
        return f'{self.name_hint}_n{len(self.remove)}_{h}'

    def transform_task(self, task, output_dir=None):
        if not self.remove and not self.goal_substitutions:
            return task
        src_path = os.path.join(
            get_libero_path('bddl_files'), task.problem_folder, task.bddl_file,
        )
        with open(src_path) as f:
            bddl = f.read()

        self._orig_order = tuple(_parse_objects_order(bddl))
        self._kept_order = tuple(k for k in self._orig_order if k not in self.remove)
        missing = set(self.remove) - set(self._orig_order)
        if missing:
            raise ValueError(f'remove keys not in :objects: {sorted(missing)}')

        bddl = _drop_section_lines(bddl, 'objects',
            lambda L: any(re.match(rf'{re.escape(k)}\s*-', L) for k in self.remove))
        bddl = _drop_section_lines(bddl, 'obj_of_interest',
            lambda L: L in self.remove)
        bddl = _drop_section_lines(bddl, 'init',
            lambda L: any(L.startswith(f'(On {k} ') for k in self.remove))

        bounds = _find_section_bounds(bddl, 'regions')
        if bounds is not None:
            s, e = bounds
            section = bddl[s:e]
            for k in self.remove:
                base = re.sub(r'_\d+$', '', k)
                section = _strip_paren_block(section, f'({base}_init_region')
            bddl = bddl[:s] + section + bddl[e:]

        if self.goal_substitutions:
            bounds = _find_section_bounds(bddl, 'goal')
            if bounds is not None:
                s, e = bounds
                section = bddl[s:e]
                for old, new in self.goal_substitutions:
                    section = section.replace(old, new)
                bddl = bddl[:s] + section + bddl[e:]

        out_dir = Path(output_dir) if output_dir else Path('/tmp')
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / f'derived__{self.slug}.bddl'
        out_path.write_text(bddl)
        self._bddl_path = str(out_path.resolve())

        if hasattr(task, '_replace'):
            return task._replace(bddl_file=self._bddl_path, problem_folder='')
        new_task = copy.copy(task)
        new_task.bddl_file = self._bddl_path
        new_task.problem_folder = ''
        return new_task

    def set_init_state(self, env, init_state, episode_idx=0):
        if not self.remove:
            return env.set_init_state(init_state)
        problem = _resolve_libero_problem_obj(env)
        if problem is None:
            return env.set_init_state(init_state)
        sim = problem.sim
        new_nq = int(sim.model.nq)
        new_nv = int(sim.model.nv)
        n_kept = len(self._kept_order)
        n_orig = len(self._orig_order)
        robot_nq = new_nq - 7 * n_kept
        robot_nv = new_nv - 6 * n_kept
        old_nq = robot_nq + 7 * n_orig
        old_nv = robot_nv + 6 * n_orig
        expected_len = 1 + old_nq + old_nv
        if len(init_state) != expected_len:
            raise ValueError(
                f'SceneRemoveObjects.set_init_state: saved init_state size '
                f'{len(init_state)} != expected {expected_len} (robot_nq={robot_nq}, '
                f'robot_nv={robot_nv}, n_orig={n_orig}). Did the BDDL change?'
            )

        qpos = list(init_state[1 : 1 + robot_nq])
        for k in self._kept_order:
            i = self._orig_order.index(k)
            start = 1 + robot_nq + 7 * i
            qpos.extend(init_state[start : start + 7])

        qvel = list(init_state[1 + old_nq : 1 + old_nq + robot_nv])
        for k in self._kept_order:
            i = self._orig_order.index(k)
            start = 1 + old_nq + robot_nv + 6 * i
            qvel.extend(init_state[start : start + 6])

        new_state = np.concatenate([[init_state[0]], qpos, qvel])
        return env.set_init_state(new_state)

    def transform_task_desc(self, desc, env=None):
        for old, new in self.prompt_replace_mapping:
            desc = desc.replace(old, new)
        return desc

    def manifest(self) -> dict:
        return {
            'kind': 'SceneRemoveObjects',
            'slug': self.slug,
            'remove': list(self.remove),
            'goal_substitutions': [list(p) for p in self.goal_substitutions],
            'prompt_replace_mapping': [list(p) for p in self.prompt_replace_mapping],
            'orig_order': list(self._orig_order),
            'kept_order': list(self._kept_order),
            'derived_bddl_path': self._bddl_path,
        }

## 1b. `_expand_pos_state_to_neg`: inverse of `SceneRemoveObjects.set_init_state`

For positive-drives we need the opposite mapping: take an `env_pos` state
(robot + kept objects only) and produce a state vector sized for `env_neg`
(robot + all 8 original objects). Robot and kept-object qpos/qvel come from
the live `env_pos` state. The 5 removed objects' qpos/qvel slots are pulled
from the episode's saved `init_state` — they stay at their initial poses
because they don't exist in `env_pos` to move.

In [ ]:
def _expand_pos_state_to_neg(pos_state_flat, init_state_flat, pos_stress, env_neg):
    """Inverse of SceneRemoveObjects.set_init_state: take a flat state vector
    sized for the kept-object scene (= env_pos.get_sim_state()) and produce a
    flat state vector sized for the original 8-object scene that can be passed
    to env_neg.set_init_state.

    Layout (LIBERO/MuJoCo):
      [time, robot_qpos, *obj_qpos_in_decl_order, robot_qvel, *obj_qvel_in_decl_order]
      with 7 qpos / 6 qvel per free-joint object.

    Removed objects (those in pos_stress.remove) get their qpos/qvel from
    init_state_flat. Kept objects + robot get theirs from pos_state_flat.
    Time and robot velocity reflect the live pos rollout, not init_state.
    """
    neg_problem = _resolve_libero_problem_obj(env_neg)
    if neg_problem is None:
        raise RuntimeError('could not resolve env_neg problem object')
    neg_nq = int(neg_problem.sim.model.nq)
    neg_nv = int(neg_problem.sim.model.nv)
    n_orig = len(pos_stress._orig_order)
    n_kept = len(pos_stress._kept_order)
    robot_nq = neg_nq - 7 * n_orig
    robot_nv = neg_nv - 6 * n_orig
    expected_pos_len = 1 + robot_nq + 7 * n_kept + robot_nv + 6 * n_kept
    expected_neg_len = 1 + neg_nq + neg_nv
    if len(pos_state_flat) != expected_pos_len:
        raise ValueError(
            f'pos_state_flat size {len(pos_state_flat)} != expected {expected_pos_len}'
        )
    if len(init_state_flat) != expected_neg_len:
        raise ValueError(
            f'init_state_flat size {len(init_state_flat)} != expected {expected_neg_len}'
        )

    out = np.asarray(init_state_flat, dtype=np.float64).copy()
    out[0] = pos_state_flat[0]  # time from live pos rollout

    # Robot qpos: from pos_state.
    out[1 : 1 + robot_nq] = pos_state_flat[1 : 1 + robot_nq]

    # Kept-object qpos: route each kept-slot in pos_state to its orig-slot in neg layout.
    for kept_idx, k in enumerate(pos_stress._kept_order):
        orig_idx = pos_stress._orig_order.index(k)
        pos_start = 1 + robot_nq + 7 * kept_idx
        neg_start = 1 + robot_nq + 7 * orig_idx
        out[neg_start : neg_start + 7] = pos_state_flat[pos_start : pos_start + 7]

    # Robot qvel: from pos_state.
    pos_qvel_robot_start = 1 + robot_nq + 7 * n_kept
    neg_qvel_robot_start = 1 + neg_nq
    out[neg_qvel_robot_start : neg_qvel_robot_start + robot_nv] = \
        pos_state_flat[pos_qvel_robot_start : pos_qvel_robot_start + robot_nv]

    # Kept-object qvel.
    for kept_idx, k in enumerate(pos_stress._kept_order):
        orig_idx = pos_stress._orig_order.index(k)
        pos_start = pos_qvel_robot_start + robot_nv + 6 * kept_idx
        neg_start = neg_qvel_robot_start + robot_nv + 6 * orig_idx
        out[neg_start : neg_start + 6] = pos_state_flat[pos_start : pos_start + 6]

    return out

## 2. Config

In [ ]:
SUITE_NAME  = 'libero_10'
TASK_ID     = 0
N_EPISODES  = 10
RESOLUTION  = 256

PROMPT = 'put both the milk and the tomato sauce in the basket'

OUT_DIR = Path('notebooks/lqr/inputs/policy_inputs') / f'{SUITE_NAME}__task{TASK_ID:02d}__milk_pos_neg'
OUT_DIR.mkdir(parents=True, exist_ok=True)
POSITIVE_NPZ  = OUT_DIR / 'positive.npz'
NEGATIVE_NPZ  = OUT_DIR / 'negative.npz'
MANIFEST_JSON = OUT_DIR / 'manifest.json'

POSITIVE_STRESS = SceneRemoveObjects(
    remove=(
        'alphabet_soup_1', 'cream_cheese_1', 'ketchup_1',
        'orange_juice_1', 'butter_1',
    ),
    goal_substitutions=(('alphabet_soup_1', 'milk_1'),),
    prompt_replace_mapping=(('alphabet soup', 'milk'),),
    name_hint='milk_only',
)
NEGATIVE_STRESS = StressTest()

DRIVE_SOURCES = (
    ('neg_drives', 0, 'env_neg drives; env_pos is state-injected (sliced)'),
    ('pos_drives', 1, 'env_pos drives; env_neg is state-injected (expanded via init_state)'),
)

print(f'suite     : {SUITE_NAME}')
print(f'task      : {TASK_ID}')
print(f'episodes  : {N_EPISODES} per drive  ({len(DRIVE_SOURCES) * N_EPISODES} total)')
print(f'prompt    : {PROMPT!r}')
print(f'positive -> {POSITIVE_NPZ.resolve()}')
print(f'negative -> {NEGATIVE_NPZ.resolve()}')

## 3. Task suite + init states + model

In [ ]:
task_suite = benchmark.get_benchmark_dict()[SUITE_NAME]()
task = task_suite.get_task(TASK_ID)
init_states = task_suite.get_task_init_states(TASK_ID)
max_env_steps = TASK_MAX_STEPS[SUITE_NAME]
print(f'task.language        : {task.language!r}')
print(f'init states available: {init_states.shape[0]}  (using the first {N_EPISODES})')
print(f'max_env_steps        : {max_env_steps}')
assert N_EPISODES <= init_states.shape[0]

In [ ]:
cfg = PolicyEvalConfig(
    config='cosmos_predict2_2b_480p_libero__inference_only',
    ckpt_path='nvidia/Cosmos-Policy-LIBERO-Predict2-2B',
    config_file='cosmos_policy/config/config.py',
    dataset_stats_path='nvidia/Cosmos-Policy-LIBERO-Predict2-2B/libero_dataset_statistics.json',
    t5_text_embeddings_path='nvidia/Cosmos-Policy-LIBERO-Predict2-2B/libero_t5_embeddings.pkl',
    use_wrist_image=True, use_proprio=True, normalize_proprio=True, unnormalize_actions=True,
    chunk_size=16, num_open_loop_steps=16, trained_with_image_aug=True,
    use_jpeg_compression=True, flip_images=True,
    num_denoising_steps_action=5,
    num_denoising_steps_future_state=1, num_denoising_steps_value=1,
    task_suite_name=SUITE_NAME,
)
dataset_stats = load_dataset_stats(cfg.dataset_stats_path)
init_t5_text_embeddings_cache(cfg.t5_text_embeddings_path)
model, _ = get_model(cfg)
print('model ready')

## 4. Build both envs

`env_neg` (unmodified scene) and `env_pos` (milk_only stripped scene) are
long-lived. Each campaign decides which one drives (steps) and which one
is state-injected.

In [ ]:
neg_task = NEGATIVE_STRESS.transform_task(task, output_dir=OUT_DIR)
env_neg, base_task_desc_neg = get_libero_env(neg_task, 'cosmos', resolution=RESOLUTION)
print(f'env_neg={type(env_neg).__name__}  base_task_desc={base_task_desc_neg!r}')

pos_task = POSITIVE_STRESS.transform_task(task, output_dir=OUT_DIR)
env_pos, base_task_desc_pos = get_libero_env(pos_task, 'cosmos', resolution=RESOLUTION)
print(f'env_pos={type(env_pos).__name__}  base_task_desc={base_task_desc_pos!r}')
print(f'orig objects: {list(POSITIVE_STRESS._orig_order)}')
print(f'kept objects: {list(POSITIVE_STRESS._kept_order)}')
print(f'derived BDDL: {POSITIVE_STRESS._bddl_path}')

## 5. Paired rollout — parameterized by which env drives

In [ ]:
def policy_fn(obs, desc):
    out = get_action(
        cfg, model, dataset_stats, obs, desc,
        num_denoising_steps_action=cfg.num_denoising_steps_action,
        generate_future_state_and_value_in_parallel=True,
    )
    return out['actions']


def _inject_into_replay(driver_role, driver_state_flat, init_state_flat,
                       env_neg, env_pos, pos_stress):
    """Given the driver env's flat state, set the replay env to the matching
    state and return its fresh obs. driver_role in {'neg', 'pos'}.
    """
    if driver_role == 'neg':
        # neg state -> env_pos: pos_stress.set_init_state slices removed slots.
        return pos_stress.set_init_state(env_pos, driver_state_flat)
    else:  # 'pos'
        # pos state -> env_neg: expand by pulling removed-object slots from init_state.
        expanded = _expand_pos_state_to_neg(
            driver_state_flat, init_state_flat, pos_stress, env_neg,
        )
        return env_neg.set_init_state(expanded)


def rollout_collect_paired(
    driver_role, init_state, env_neg, env_pos, pos_stress, prompt,
    *, num_steps_wait=10,
):
    """One paired rollout under the chosen driver. Returns
    (success, env_steps, neg_inputs, pos_inputs). neg/pos lists are always
    indexed by *scene* (which set of objects is present in the rendered
    image), not by which env drove the rollout. Both lists have the same
    length and are paired row-for-row.
    """
    if driver_role == 'neg':
        driver_env, replay_env = env_neg, env_pos
    elif driver_role == 'pos':
        driver_env, replay_env = env_pos, env_neg
    else:
        raise ValueError(f'driver_role must be \'neg\' or \'pos\', got {driver_role!r}')

    # Reset + initialize driver. For env_pos as driver, pos_stress slices init_state.
    driver_env.reset()
    if driver_role == 'neg':
        obs_driver = driver_env.set_init_state(init_state)
    else:
        obs_driver = pos_stress.set_init_state(driver_env, init_state)

    # Initialize replay env to a sane starting state too (overwritten per inference).
    if driver_role == 'neg':
        pos_stress.set_init_state(replay_env, init_state)
    else:
        replay_env.set_init_state(init_state)

    # Settle driver
    for _ in range(num_steps_wait):
        obs_driver, _, _, _ = driver_env.step(get_libero_dummy_action(cfg.model_family))

    queue = deque(maxlen=cfg.num_open_loop_steps)
    neg_inputs, pos_inputs = [], []
    success = False
    t = 0
    while t < max_env_steps:
        if not queue:
            obs_driver_packed = prepare_observation(
                obs_driver, resize_size=224, flip_images=cfg.flip_images,
            )

            # Snapshot driver state, inject into replay env, pack its obs.
            driver_state_flat = driver_env.get_sim_state()
            obs_replay = _inject_into_replay(
                driver_role, driver_state_flat, init_state,
                env_neg, env_pos, pos_stress,
            )
            obs_replay_packed = prepare_observation(
                obs_replay, resize_size=224, flip_images=cfg.flip_images,
            )

            # Sort driver/replay into neg/pos by scene identity.
            if driver_role == 'neg':
                neg_packed, pos_packed = obs_driver_packed, obs_replay_packed
            else:
                pos_packed, neg_packed = obs_driver_packed, obs_replay_packed

            neg_inputs.append({
                'primary_image': np.ascontiguousarray(neg_packed['primary_image']),
                'wrist_image':   np.ascontiguousarray(neg_packed['wrist_image']),
                'proprio':       np.asarray(neg_packed['proprio'], dtype=np.float32),
            })
            pos_inputs.append({
                'primary_image': np.ascontiguousarray(pos_packed['primary_image']),
                'wrist_image':   np.ascontiguousarray(pos_packed['wrist_image']),
                'proprio':       np.asarray(pos_packed['proprio'], dtype=np.float32),
            })

            # Drive: policy runs on the driver env's obs (whichever scene that is).
            actions = policy_fn(obs_driver_packed, prompt)
            for a in actions[:cfg.num_open_loop_steps]:
                queue.append(np.asarray(a, dtype=np.float32))

        a = queue.popleft()
        obs_driver, _, done, _ = driver_env.step(a.tolist())
        if done:
            success = True
            break
        t += 1
    return success, t + num_steps_wait, neg_inputs, pos_inputs

## 6. Run both drive campaigns

Negative-drives first (env_neg steps), then positive-drives (env_pos steps).
All rows from both campaigns land in the same `positive.npz` / `negative.npz`,
distinguished by `drive_source` (0=neg-drives, 1=pos-drives).

In [ ]:
all_pos_primary, all_pos_wrist, all_pos_proprio = [], [], []
all_neg_primary, all_neg_wrist, all_neg_proprio = [], [], []
all_episode_idx, all_inference_idx, all_drive_source = [], [], []
rollout_summaries = []

for drive_name, drive_code, drive_desc in DRIVE_SOURCES:
    role = 'neg' if drive_name == 'neg_drives' else 'pos'
    print(f'=== {drive_name}  (drive_source={drive_code}) — {drive_desc} ===')
    for ep in range(N_EPISODES):
        t0 = time.time()
        success, env_steps, neg_inputs, pos_inputs = rollout_collect_paired(
            role, init_states[ep], env_neg, env_pos, POSITIVE_STRESS, PROMPT,
        )
        dt = time.time() - t0
        assert len(neg_inputs) == len(pos_inputs), 'paired-row invariant violated'

        for inf_idx, (n_rec, p_rec) in enumerate(zip(neg_inputs, pos_inputs)):
            all_neg_primary.append(n_rec['primary_image'])
            all_neg_wrist.append(n_rec['wrist_image'])
            all_neg_proprio.append(n_rec['proprio'])
            all_pos_primary.append(p_rec['primary_image'])
            all_pos_wrist.append(p_rec['wrist_image'])
            all_pos_proprio.append(p_rec['proprio'])
            all_episode_idx.append(ep)
            all_inference_idx.append(inf_idx)
            all_drive_source.append(drive_code)

        tag = 'SUCCESS' if success else 'FAILURE'
        n_inf = len(neg_inputs)
        print(f'  ep {ep:2d}  {tag:7s}  steps={env_steps:4d}  inferences={n_inf:3d}  {dt:6.1f}s')
        rollout_summaries.append({
            'drive_source': drive_code,
            'drive_name': drive_name,
            'episode': ep,
            'success': bool(success),
            'env_steps': int(env_steps),
            'n_inferences': n_inf,
            'wall_time_s': dt,
        })

env_neg.close()
env_pos.close()
print()
print(f'total paired rows: {len(all_episode_idx)}')

## 7. Save paired NPZs

Both files share row count and the same `(episode_idx, inference_idx, drive_source)`
columns. Row `i` in `positive.npz` and row `i` in `negative.npz` share the
same proprio by construction (same MuJoCo state at capture time) and differ
only in rendered camera content.

In [ ]:
def _stack_save(out_npz, primary_list, wrist_list, proprio_list):
    primary_arr   = np.stack(primary_list, axis=0)
    wrist_arr     = np.stack(wrist_list,   axis=0)
    proprio_arr   = np.stack(proprio_list, axis=0)
    episode_arr   = np.asarray(all_episode_idx,   dtype=np.int32)
    inference_arr = np.asarray(all_inference_idx, dtype=np.int32)
    drive_arr     = np.asarray(all_drive_source,  dtype=np.int32)
    np.savez_compressed(
        out_npz,
        primary_images=primary_arr,
        wrist_images=wrist_arr,
        proprios=proprio_arr,
        episode_idx=episode_arr,
        inference_idx=inference_arr,
        drive_source=drive_arr,
    )
    size_mb = out_npz.stat().st_size / 1e6
    print(f'wrote {out_npz}  ({size_mb:.1f} MB)')
    print(f'  primary_images: {primary_arr.shape}  dtype={primary_arr.dtype}')
    print(f'  wrist_images:   {wrist_arr.shape}  dtype={wrist_arr.dtype}')
    print(f'  proprios:       {proprio_arr.shape}  dtype={proprio_arr.dtype}')
    return primary_arr.shape[0]


n_pos = _stack_save(POSITIVE_NPZ, all_pos_primary, all_pos_wrist, all_pos_proprio)
n_neg = _stack_save(NEGATIVE_NPZ, all_neg_primary, all_neg_wrist, all_neg_proprio)
assert n_pos == n_neg, 'paired-row invariant violated after save'

pos_proprio = np.stack(all_pos_proprio, axis=0)
neg_proprio = np.stack(all_neg_proprio, axis=0)
max_dproprio = float(np.max(np.abs(pos_proprio - neg_proprio)))
print(f'paired proprio max |diff|: {max_dproprio:.3e}  (should be ~0)')

# Per-drive breakdown of paired-row counts
drive_arr_all = np.asarray(all_drive_source, dtype=np.int32)
for name, code_id, _ in DRIVE_SOURCES:
    n = int((drive_arr_all == code_id).sum())
    print(f'  drive_source={code_id} ({name}): {n} rows')

## 8. Visualize a few paired examples

Sanity-check the contrastive setup: a few evenly-spaced inference rows from
one episode per `drive_source`, side-by-side (neg-scene render | pos-scene
render) for both the primary (agentview) and wrist cameras. Within a row,
the two scenes should differ only in distractor presence — robot pose and
kept-object positions are identical by construction.

In [ ]:
import matplotlib.pyplot as plt

VIZ_EPISODE          = 0   # which episode to draw from (per drive_source)
VIZ_SAMPLES_PER_DRIVE = 3  # how many inference timesteps to show per drive_source

drive_arr_all     = np.asarray(all_drive_source,   dtype=np.int32)
episode_arr_all   = np.asarray(all_episode_idx,    dtype=np.int32)
inference_arr_all = np.asarray(all_inference_idx,  dtype=np.int32)

def _pick_rows(drive_code, episode, n_samples):
    """Return up to n_samples row indices, evenly spaced through the chosen
    episode of the chosen drive_source."""
    mask = (drive_arr_all == drive_code) & (episode_arr_all == episode)
    rows = np.where(mask)[0]
    if len(rows) == 0:
        return np.array([], dtype=int)
    if len(rows) <= n_samples:
        return rows
    return rows[np.linspace(0, len(rows) - 1, n_samples).astype(int)]

selected = []
for drive_name, drive_code, _ in DRIVE_SOURCES:
    for i in _pick_rows(drive_code, VIZ_EPISODE, VIZ_SAMPLES_PER_DRIVE):
        selected.append((drive_name, drive_code, int(i)))

if not selected:
    print(f'no rows for episode {VIZ_EPISODE}; nothing to visualize')
else:
    n_rows = len(selected)
    fig, axes = plt.subplots(n_rows, 4, figsize=(13, 3.2 * n_rows))
    if n_rows == 1:
        axes = axes[np.newaxis, :]

    for row_i, (drive_name, drive_code, i) in enumerate(selected):
        inf_idx = int(inference_arr_all[i])
        dproprio = float(np.max(np.abs(all_pos_proprio[i] - all_neg_proprio[i])))
        row_tag = f'{drive_name}  ep{VIZ_EPISODE}  inf{inf_idx:3d}'

        axes[row_i, 0].imshow(all_neg_primary[i])
        axes[row_i, 0].set_title(f'{row_tag}\nneg primary  (distractors present)', fontsize=9)
        axes[row_i, 0].axis('off')

        axes[row_i, 1].imshow(all_pos_primary[i])
        axes[row_i, 1].set_title('pos primary  (distractors absent)', fontsize=9)
        axes[row_i, 1].axis('off')

        axes[row_i, 2].imshow(all_neg_wrist[i])
        axes[row_i, 2].set_title('neg wrist', fontsize=9)
        axes[row_i, 2].axis('off')

        axes[row_i, 3].imshow(all_pos_wrist[i])
        axes[row_i, 3].set_title(f'pos wrist   |Δproprio|∞={dproprio:.1e}', fontsize=9)
        axes[row_i, 3].axis('off')

    plt.tight_layout()
    plt.show()

## 9. Manifest

In [ ]:
manifest = {
    'suite': SUITE_NAME,
    'task_id': TASK_ID,
    'n_episodes_per_drive': N_EPISODES,
    'resolution': RESOLUTION,
    'prompt': PROMPT,
    'pairing': (
        'row i in positive.npz and row i in negative.npz share the same '
        'MuJoCo state at capture time. drive_source distinguishes which env '
        'stepped: 0 = env_neg drove, env_pos was state-injected (sliced via '
        'SceneRemoveObjects); 1 = env_pos drove, env_neg was state-injected '
        '(expanded via _expand_pos_state_to_neg, removed-object slots pulled '
        'from the episode init_state).'
    ),
    'image_layout': 'HWC uint8, flip_images=True applied at capture time (same as get_action input)',
    'proprio_layout': 'concat(robot0_gripper_qpos[2], robot0_eef_pos[3], robot0_eef_quat[4]) -> shape (9,) float32',
    'drive_sources': [
        {'code': code_id, 'name': name, 'desc': desc}
        for name, code_id, desc in DRIVE_SOURCES
    ],
    'sets': {
        'positive': {
            'out_npz': str(POSITIVE_NPZ),
            'stress_test': POSITIVE_STRESS.manifest(),
            'env_base_task_desc': base_task_desc_pos,
            'role': 'distractors-absent render at every captured pose',
        },
        'negative': {
            'out_npz': str(NEGATIVE_NPZ),
            'stress_test': NEGATIVE_STRESS.manifest(),
            'env_base_task_desc': base_task_desc_neg,
            'role': 'distractors-present render at every captured pose',
        },
    },
    'rollouts': rollout_summaries,
    'paired_proprio_max_abs_diff': max_dproprio,
    'positive_drives_caveat': (
        'in positive-drives rows (drive_source=1), the removed distractors '
        'in the rendered negative image are at their *initial* poses — they '
        'never physically interacted with the robot, because the robot was '
        'actually stepping in env_pos where those objects don\'t exist. '
        'Visually this is fine for "distractors present vs absent" semantics, '
        'but it is a physical inconsistency (the robot may have visually '
        'passed through where a distractor would have been).'
    ),
}
MANIFEST_JSON.write_text(json.dumps(manifest, indent=2))
print(f'wrote {MANIFEST_JSON}')

## 10. Summary

In [ ]:
for name, code_id, _ in DRIVE_SOURCES:
    subset = [r for r in rollout_summaries if r['drive_source'] == code_id]
    n_succ = sum(r['success'] for r in subset)
    n_inf = sum(r['n_inferences'] for r in subset)
    print(f'  {name:11s}  success={n_succ}/{len(subset)}  rows={n_inf}')
print(f'  total paired rows           : {len(all_episode_idx)}')
print(f'  paired proprio max |diff|   : {max_dproprio:.3e}')
print(f'  positive (distractors absent ): {POSITIVE_NPZ.resolve()}')
print(f'  negative (distractors present): {NEGATIVE_NPZ.resolve()}')
print(f'  manifest                     : {MANIFEST_JSON.resolve()}')